# Corporate Financial Anomaly Detection System

This notebook is designed to run on Google Colab. It processes 492 Nifty 500 company financial spreadsheets, extracts key fundamentals, and uses Unsupervised Machine Learning (Isolation Forests) to flag potential accounting anomalies or severe financial distress.

## Step 1: Upload Data
Upload the `screener_exports.zip` file (generated by `prepare_colab_data.py`) to your Colab session. Then run the cell below to extract it.

In [ ]:
!unzip -q screener_exports.zip -d screener_exports
print('Data unzipped successfully!')

## Step 2: Data Ingestion Pipeline
We iterate through all the Excel files, find the `Data Sheet`, and extract time-series financial data.

In [ ]:
import os
import glob
import pandas as pd
import numpy as np

def extract_financials(file_path):
    symbol = os.path.basename(file_path).replace('.xlsx', '')
    try:
        df = pd.read_excel(file_path, sheet_name='Data Sheet')
    except Exception:
        return []
    
    industry = 'Unknown'
    for idx, row in df.iterrows():
        if str(row.iloc[0]).strip() == 'Industry':
            industry = row.iloc[1]
            break

    sections = ['PROFIT & LOSS', 'BALANCE SHEET', 'CASH FLOW']
    data_records = {}
    current_section = None
    dates = []
    
    for idx, row in df.iterrows():
        metric = str(row.iloc[0]).strip()
        if metric in sections:
            current_section = metric
            continue
        if current_section and metric == 'Report Date':
            dates = []
            for col_idx in range(1, len(row)):
                val = row.iloc[col_idx]
                if pd.notna(val) and (isinstance(val, pd.Timestamp) or str(val).startswith('20')):
                    if isinstance(val, pd.Timestamp):
                        dates.append((col_idx, val.year))
                    else:
                        dates.append((col_idx, str(val)[:4]))
            continue
        if current_section and pd.notna(metric) and metric != 'nan':
            for col_idx, date_label in dates:
                val = row.iloc[col_idx]
                if pd.notna(val):
                    key = (symbol, date_label)
                    if key not in data_records:
                        data_records[key] = {'Company': symbol, 'Year': date_label, 'Industry': industry}
                    data_records[key][metric] = val
    return list(data_records.values())

files = glob.glob('screener_exports/*.xlsx')
all_data = []
print(f'Processing {len(files)} files...')
for i, f in enumerate(files):
    all_data.extend(extract_financials(f))
    if (i+1) % 100 == 0:
        print(f'Processed {i+1}/{len(files)}')

df_raw = pd.DataFrame(all_data)
print(f'\nRaw panel data shape: {df_raw.shape}')

## Step 3: Feature Engineering
Calculating Machine Learning features: Accruals Ratio, Leverage Change, and Margin Volatility.

In [ ]:
df = df_raw.copy()
df.columns = [str(c).strip() for c in df.columns]
df.sort_values(by=['Company', 'Year'], inplace=True)

features = df[['Company', 'Year', 'Industry']].copy()

# 1. Accruals Ratio
if 'Net profit' in df.columns and 'Cash from Operating Activity' in df.columns and 'Total' in df.columns:
    np_val = pd.to_numeric(df['Net profit'], errors='coerce')
    cfo_val = pd.to_numeric(df['Cash from Operating Activity'], errors='coerce')
    assets_val = pd.to_numeric(df['Total'], errors='coerce')
    features['Accruals_Ratio'] = (np_val - cfo_val) / assets_val.replace(0, np.nan)

# 2. Leverage Change YoY
debt_col = next((c for c in df.columns if 'Borrowings' in c or 'Debt' in c), None)
if debt_col and 'Total' in df.columns:
    debt_val = pd.to_numeric(df[debt_col], errors='coerce')
    assets_val = pd.to_numeric(df['Total'], errors='coerce')
    features['Leverage_Ratio'] = debt_val / assets_val.replace(0, np.nan)
    features['Leverage_Change_YoY'] = features.groupby('Company')['Leverage_Ratio'].diff()

# 3. Margin Volatility
sales_col = next((c for c in df.columns if 'Sales' in c or 'Revenue' in c), None)
op_profit_col = next((c for c in df.columns if 'Operating Profit' in c or 'EBITDA' in c), None)
if sales_col and op_profit_col:
    sales_val = pd.to_numeric(df[sales_col], errors='coerce')
    op_val = pd.to_numeric(df[op_profit_col], errors='coerce')
    features['Operating_Margin'] = op_val / sales_val.replace(0, np.nan)
    features['Margin_Volatility_3yr'] = features.groupby('Company')['Operating_Margin'].transform(lambda x: x.rolling(3, min_periods=2).std())

ml_cols = ['Accruals_Ratio', 'Leverage_Change_YoY', 'Margin_Volatility_3yr']
features.dropna(subset=ml_cols, how='all', inplace=True)
print(f'ML Features shape: {features.shape}')

## Step 4: Machine Learning (Isolation Forests)
Training the anomaly detection model.

In [ ]:
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler

df_ml = features.copy()
df_ml[ml_cols] = df_ml.groupby('Industry')[ml_cols].transform(lambda x: x.fillna(x.median()))
df_ml[ml_cols] = df_ml[ml_cols].fillna(df_ml[ml_cols].median())

df_ml['Anomaly_Score'] = np.nan
df_ml['Is_Anomaly'] = -1

for industry, group in df_ml.groupby('Industry'):
    if len(group) > 10:
        X = group[ml_cols]
        scaler = StandardScaler()
        X_scaled = scaler.fit_transform(X)
        clf = IsolationForest(contamination=0.05, random_state=42)
        clf.fit(X_scaled)
        df_ml.loc[group.index, 'Anomaly_Score'] = clf.decision_function(X_scaled) * -1
        df_ml.loc[group.index, 'Is_Anomaly'] = (clf.predict(X_scaled) == -1).astype(int)

unscored = df_ml['Anomaly_Score'].isna()
if unscored.any():
    X = df_ml.loc[unscored, ml_cols]
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)
    clf = IsolationForest(contamination=0.05, random_state=42)
    clf.fit(X_scaled)
    df_ml.loc[unscored, 'Anomaly_Score'] = clf.decision_function(X_scaled) * -1
    df_ml.loc[unscored, 'Is_Anomaly'] = (clf.predict(X_scaled) == -1).astype(int)

df_ml.sort_values(by='Anomaly_Score', ascending=False, inplace=True)
df_ml.to_csv('anomaly_scores.csv', index=False)
print('Model training complete! Download anomaly_scores.csv to your local machine.')